# QA Agent — 2026 Anthem HIPSBC Plan

PDF-based Q&A using `ChatOpenAI` + `FAISS` + `LangChain`

In [ ]:
# ── 1. Load environment variables ──────────────────────────────────────────
import os
from dotenv import load_dotenv
from pathlib import Path
from langchain_openai import ChatOpenAI

env_path = Path(".env")
load_dotenv(env_path)

OPENAI_API_KEY = os.getenv("UNIFIED_LLM_KEY")
os.environ['ANTHROPIC_API_KEY'] = OPENAI_API_KEY

print("API key loaded:", bool(OPENAI_API_KEY))

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage
import base64

# ── Read PDF as base64 ─────────────────────────────────────────────────────
pdf_path = "data/2026AnthemgHIPSBC.pdf"

with open(pdf_path, "rb") as f:
    pdf_base64 = base64.standard_b64encode(f.read()).decode("utf-8")

print(f"PDF loaded — base64 length: {len(pdf_base64):,} chars")

# ── Init ChatAnthropic ─────────────────────────────────────────────────────
model = ChatAnthropic(model="claude-opus-4-6")

# ── Build message with document block ─────────────────────────────────────
question = "What is the deductible and out-of-pocket maximum for the 2026 Anthem HIPSBC plan?"

message = HumanMessage(
    content=[
        {
            "type": "document",
            "source": {
                "type":       "base64",
                "media_type": "application/pdf",
                "data":       pdf_base64,
            },
        },
        {
            "type": "text",
            "text": question,
        },
    ]
)

# ── Invoke ─────────────────────────────────────────────────────────────────
response = model.invoke([message])
print("\nQuestion:", question)
print("\nAnswer:\n", response.content)

## 3.4. Refactor into an Agent Class

To make this code reusable and easier to integrate into an A2A server later, you will wrap the logic into a `PolicyAgent` class in a file named `agents.py`. This class initializes the client and data in the `__init__` method and exposes an `answer_query` method.

In [ ]:
%%writefile agents.py
import os
import base64
from dotenv import load_dotenv
from pathlib import Path
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

load_dotenv(Path(".env"))
os.environ["ANTHROPIC_API_KEY"] = os.getenv("UNIFIED_LLM_KEY", "")


class PolicyAgent:
    def __init__(self, model: str = "claude-opus-4-6"):
        pdf_path= "/Users/vinotganesan/Learning/LLM & AGENTS/A2A/deeplearning.ai/data/2026AnthemgHIPSBC.pdf"
        with open(pdf_path, "rb") as f:
            self.pdf_base64 = base64.standard_b64encode(f.read()).decode("utf-8")

        print(f"PDF loaded — base64 length: {len(self.pdf_base64):,} chars")

        self.model = ChatAnthropic(model=model)

    def answer_query(self, question: str) -> str:
        message = HumanMessage(
            content=[
                {
                    "type": "document",
                    "source": {
                        "type":       "base64",
                        "media_type": "application/pdf",
                        "data":       self.pdf_base64,
                    },
                },
                {
                    "type": "text",
                    "text": question,
                },
            ]
        )

        response = self.model.invoke([message])
        return response.content


if __name__ == "__main__":
    agent = PolicyAgent()

    question = "What is the deductible and out-of-pocket maximum for the 2026 Anthem HIPSBC plan?"
    print("\nQuestion:", question)
    print("\nAnswer:\n", agent.answer_query(question))

In [ ]:
import importlib
import agents
importlib.reload(agents)

from agents import PolicyAgent
from IPython.display import Markdown, display

print("Running Health Insurance Policy Agent")
agent = PolicyAgent()
prompt = "How much would I pay for mental health therapy?"

response = agent.answer_query(prompt)
display(Markdown(response))